<a href="https://colab.research.google.com/github/192524158simats-gif/ITA0616--Machine-learning/blob/main/Lab%20Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import math
from collections import Counter

# Play Tennis Dataset
data = [
    {'Outlook':'Sunny', 'Temperature':'Hot', 'Humidity':'High', 'Wind':'Weak', 'Play':'No'},
    {'Outlook':'Sunny', 'Temperature':'Hot', 'Humidity':'High', 'Wind':'Strong', 'Play':'No'},
    {'Outlook':'Overcast', 'Temperature':'Hot', 'Humidity':'High', 'Wind':'Weak', 'Play':'Yes'},
    {'Outlook':'Rain', 'Temperature':'Mild', 'Humidity':'High', 'Wind':'Weak', 'Play':'Yes'},
    {'Outlook':'Rain', 'Temperature':'Cool', 'Humidity':'Normal', 'Wind':'Weak', 'Play':'Yes'},
    {'Outlook':'Rain', 'Temperature':'Cool', 'Humidity':'Normal', 'Wind':'Strong', 'Play':'No'},
    {'Outlook':'Overcast', 'Temperature':'Cool', 'Humidity':'Normal', 'Wind':'Strong', 'Play':'Yes'},
    {'Outlook':'Sunny', 'Temperature':'Mild', 'Humidity':'High', 'Wind':'Weak', 'Play':'No'},
    {'Outlook':'Sunny', 'Temperature':'Cool', 'Humidity':'Normal', 'Wind':'Weak', 'Play':'Yes'},
    {'Outlook':'Rain', 'Temperature':'Mild', 'Humidity':'Normal', 'Wind':'Weak', 'Play':'Yes'},
    {'Outlook':'Sunny', 'Temperature':'Mild', 'Humidity':'Normal', 'Wind':'Strong', 'Play':'Yes'},
    {'Outlook':'Overcast', 'Temperature':'Mild', 'Humidity':'High', 'Wind':'Strong', 'Play':'Yes'},
    {'Outlook':'Overcast', 'Temperature':'Hot', 'Humidity':'Normal', 'Wind':'Weak', 'Play':'Yes'},
    {'Outlook':'Rain', 'Temperature':'Mild', 'Humidity':'High', 'Wind':'Strong', 'Play':'No'}
]

# Calculate Entropy
def entropy(rows):
    counts = Counter(row['Play'] for row in rows)
    total = len(rows)

    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values()
    )

# Calculate Information Gain
def information_gain(rows, attribute):
    total_entropy = entropy(rows)
    total = len(rows)

    values = set(row[attribute] for row in rows)

    weighted_entropy = 0

    for value in values:
        subset = [row for row in rows if row[attribute] == value]

        weighted_entropy += (
            len(subset) / total
        ) * entropy(subset)

    return total_entropy - weighted_entropy

# Build ID3 Decision Tree
def id3(rows, attributes):

    labels = [row['Play'] for row in rows]

    # If all examples have the same class
    if len(set(labels)) == 1:
        return labels[0]

    # If no attributes remain
    if not attributes:
        return Counter(labels).most_common(1)[0][0]

    # Select attribute with maximum information gain
    best_attribute = max(
        attributes,
        key=lambda a: information_gain(rows, a)
    )

    tree = {best_attribute: {}}

    remaining = [a for a in attributes if a != best_attribute]

    for value in sorted(set(row[best_attribute] for row in rows)):

        subset = [
            row for row in rows
            if row[best_attribute] == value
        ]

        tree[best_attribute][value] = id3(subset, remaining)

    return tree

# Print Decision Tree
def print_tree(tree, indent=""):

    if not isinstance(tree, dict):
        print(indent + "-> " + tree)
        return

    attribute = next(iter(tree))

    for value, branch in tree[attribute].items():
        print(indent + attribute + " = " + value)
        print_tree(branch, indent + "  ")

# Predict a new sample
def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    if value not in tree[attribute]:
        return "Unknown"

    return predict(tree[attribute][value], sample)


# Main Program
attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']

print("Initial Entropy:", round(entropy(data), 3))

print("\nInformation Gain:")
for attribute in attributes:
    print(attribute, ":", round(information_gain(data, attribute), 3))

decision_tree = id3(data, attributes)

print("\nDecision Tree:")
print_tree(decision_tree)

# New sample classification
new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print("\nNew Sample:", new_sample)
print("Predicted Result:", predict(decision_tree, new_sample))

Initial Entropy: 0.94

Information Gain:
Outlook : 0.247
Temperature : 0.029
Humidity : 0.152
Wind : 0.048

Decision Tree:
Outlook = Overcast
  -> Yes
Outlook = Rain
  Wind = Strong
    -> No
  Wind = Weak
    -> Yes
Outlook = Sunny
  Humidity = High
    -> No
  Humidity = Normal
    -> Yes

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Result: No
